# Agent | Tool

- Chain
    - 개발자가 실행 흐름을 미리 정한다.
    - 예: 프롬프트 -> 모델 -> 출력 파서

- Agent
    - 모델이 사용자의 요청을 보고 필요한 도구를 선택한다.
    - 도구 실행 결과를 보고 다시 판단한다.
    - 필요한 경우 여러 번 도구를 호출한 뒤 최종 답변을 만든다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool

In [3]:
llm = init_chat_model("openai:gpt-4.1-mini",temperature = 0)

## 간단한 tool 만들기
- @tool 데코레이터를 붙여주면 LangChain Agent가 호출할 수 있는 Tool이 된다.

In [10]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result"""
    return a * b

In [11]:
# Tool을 invoke() 함수를 통해 직접 실행할 수도 있다.
multiply.invoke({
    'a':12,
    'b':7
})

84

In [ ]:
# Tool 객체에는 이름, 설명, 입력 스키마가 포함된다.
# Agent는 이 정보를 보고 어떤 도구를 호출할지 판단한다.
print("Tool name: ", multiply.name)
print("Tool description: ", multiply.description)
print("Tool args: ", multiply.args)

Tool name:  multiply
Tool description:  Multiply two integers and return the result
Tool args:  {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


## Tool descroption 품질 비교

In [7]:
@tool("bad_calc")
def bad_calc(price:int,quantity:int,tax_rate:float) -> int:
    """
    Caculate something
    """
    return price * quantity * (1+tax_rate)

@tool("calculate_total_price")
def calculate_total_price(price:int,quantity:int,tax_rate:float) -> int:
    """
    Calculate the final total price for a product order.
    Use this when the user provides an item price, quantity, and tax rate.
    The tax_rate should be given as a decimal value, such as 0.1 for 10%.
    """
    return price * quantity * (1+tax_rate)

In [8]:
docstring_agent = create_agent(
    model=llm,
    tools=[bad_calc,calculate_total_price],
    system_prompt="""
당신은 계산을 도와주는 AI 도우미입니다.
사용자의 요청에 가장 적합한 계산 도구를 선택하세요.
최종 답변은 한국어로 작성하세요.
"""
)

result = docstring_agent.invoke({
    "messages" : [
        {
            "role":"user",
            "content":"12500원짜리 상품을 3개 살 때 부가세 10%를 포함한 최종 금액은 얼마인가요?"
        }
    ]
})

result['messages'][-1].content

'12500원짜리 상품 3개를 부가세 10% 포함하여 구매할 때 최종 금액은 41,250원입니다.'

In [9]:
# 어떤 tool을 호출 했는지 메세지 흐름을 확인한다.
for messages in result['messages']:
    print(type(messages).__name__)
    print(messages)
    print("-"*80)

HumanMessage
content='12500원짜리 상품을 3개 살 때 부가세 10%를 포함한 최종 금액은 얼마인가요?' additional_kwargs={} response_metadata={} id='a435dde6-b0ed-4103-8b27-99d31e86cb47'
--------------------------------------------------------------------------------
AIMessage
content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 192, 'total_tokens': 219, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_393befd1a4', 'id': 'chatcmpl-DeXMgP6E3wJDnJNLRSGpCZBJGjdDR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e1a10-6cf9-7f33-9a94-54f370e1ce74-0' tool_calls=[{'name': 'calculate_total_price', 'args': {'price': 12500, 'quantity': 3, 'tax_rate': 0.1}, 'id': 'call

## 여러 tool 연결하기

In [12]:
@tool
def add(a:int,b:int)->int:
    """Add two integers add return the result"""
    return a + b

@tool
def divide(a:int,b:int) -> int:
    """Divide a by b and return the result. The dividor b must not be zero"""
    if b == 0:
        raise ValueError("b must not be zero")
    return a / b

In [13]:
math_agent = create_agent(
    model=llm,
    tools=[add, multiply, divide],
    system_prompt="""
당신은 계산을 도롸주는 AI도우미입니다.
계산이 필요한 경우 제공된 도구를 사용하세요.
최종 답변은 한국어로 작성하세요.
"""
)

In [14]:
result  = math_agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"15와 8을 더한 뒤, 그  결과에 3을 곱하면 얼마인가요?"
        }
    ]
})

result['messages'][-1].content

'15와 8을 더한 결과는 23이고, 그 결과에 3을 곱하면 69입니다.'

## Tool 실행 실패 처리하기

In [15]:
import json

@tool("safe_divide")
def safe_divide(a:float,b:float) -> str:
    """
    Divide a by b safely.
    Use this when the users asks for division.
    If b is zero, return a structed error message instead of raising an exception
    """
    if b==0:
        return json.dumps({
            "ok" : False,
            "error" : "0으로 나눌 수 없습니다.",
            "a":a,
            "b":b
        },ensure_ascii=False)
    
    
    return json.dumps({
        "ok" : True,
        "result" : a / b,
        "a":a,
        "b":b
    },ensure_ascii=False)

In [16]:
# 실패 상황을 Agent 연결하기 전 Tool 단독 호출로 언저 확인
print(safe_divide.invoke({"a":10,"b":2}))
print(safe_divide.invoke({"a":10,"b":0}))

{"ok": true, "result": 5.0, "a": 10.0, "b": 2.0}
{"ok": false, "error": "0으로 나눌 수 없습니다.", "a": 10.0, "b": 0.0}


In [18]:
safe_math_agent = create_agent(
    model=llm,
    tools=[safe_divide],
    system_prompt="""
당신은 계산을 도와주는 도우미 AI입니다.
safe_divide 도구 결과의 ok 값이 False이면, 계산할 수 없는 이유를 사용자에게 설명하세요.
최종 답변은 한국어로 작성하세요.
"""
)

result = safe_math_agent.invoke({
    "messages":[
        {
            "role" : "user",
            "content" : "10을 0으로 나누면 얼마인가요?"
        }
    ]
})

result['messages'][-1].content

'10을 0으로 나누는 것은 수학적으로 정의되지 않아서 계산할 수 없습니다. 0으로 나누는 것은 불가능합니다.'

# 날씨 tool 만들기

In [19]:
WEATHER_CODE_MAP = {
    0: "맑음",
    1: "대체로 맑음",
    2: "부분적으로 흐림",
    3: "흐림",
    45: "안개",
    48: "서리 안개",
    51: "약한 이슬비",
    53: "이슬비",
    55: "강한 이슬비",
    61: "약한 비",
    63: "비",
    65: "강한 비",
    71: "약한 눈",
    73: "눈",
    75: "강한 눈",
    80: "약한 소나기",
    81: "소나기",
    82: "강한 소나기",
    95: "뇌우",
    96: "약한 우박을 동반한 뇌우",
    99: "강한 우박을 동반한 뇌우",
}

In [20]:
import requests

@tool("current_weather")
def get_current_weather(city: str) -> str:
    """
    Get current weather for a city using Open-Meteo. 
    No API key is required. 
    The city input should be written in English, such as Seoul, Busan, Tokyo, or New York.
    """
    
    # 1. 도시명을 위도/경도로 변환한다.
    geocoding_url = "https://geocoding-api.open-meteo.com/v1/search"

    geocoding_params = {
        "name": city,
        "count": 1,
        "language": "ko",
        "format": "json",
    }

    try:
        geocoding_response = requests.get(
            geocoding_url,
            params=geocoding_params,
            timeout=5,
        )
        geocoding_response.raise_for_status()
        geocoding_data = geocoding_response.json()
    except requests.RequestException as error:
        return json.dumps(
            {
                "found": False,
                "error": f"도시 정보를 조회하는 중 오류가 발생했습니다: {error}",
            },
            ensure_ascii=False,
        )

    results = geocoding_data.get("results", [])

    if not results:
        return json.dumps(
            {
                "found": False,
                "error": f"'{city}'에 해당하는 도시를 찾지 못했습니다.",
            },
            ensure_ascii=False,
        )

    location = results[0]

    latitude = location["latitude"]
    longitude = location["longitude"]
    city_name = location.get("name", city)
    country = location.get("country", "")
    admin1 = location.get("admin1", "")

    # 2. 위도/경도를 사용해 현재 날씨를 조회한다.
    weather_url = "https://api.open-meteo.com/v1/forecast"

    weather_params = {
        "latitude": latitude,
        "longitude": longitude,
        "current": ",".join([
            "temperature_2m",
            "relative_humidity_2m",
            "apparent_temperature",
            "precipitation",
            "weather_code",
            "wind_speed_10m",
        ]),
        "timezone": "auto",
    }

    try:
        weather_response = requests.get(
            weather_url,
            params=weather_params,
            timeout=5,
        )
        weather_response.raise_for_status()
        weather_data = weather_response.json()
    except requests.RequestException as error:
        return json.dumps(
            {
                "found": False,
                "error": f"날씨 정보를 조회하는 중 오류가 발생했습니다: {error}",
            },
            ensure_ascii=False,
        )

    current = weather_data.get("current", {})
    weather_code = current.get("weather_code")

    weather_info = {
        "found": True,
        "city": city_name,
        "admin1": admin1,
        "country": country,
        "latitude": latitude,
        "longitude": longitude,
        "time": current.get("time"),
        "weather": WEATHER_CODE_MAP.get(weather_code, f"알 수 없음({weather_code})"),
        "temperature_celsius": current.get("temperature_2m"),
        "apparent_temperature_celsius": current.get("apparent_temperature"),
        "relative_humidity_percent": current.get("relative_humidity_2m"),
        "precipitation_mm": current.get("precipitation"),
        "wind_speed_kmh": current.get("wind_speed_10m"),
    }

    # Tool의 반환값은 모델이 읽을 수 있도록 문자열로 반환한다.
    return json.dumps(weather_info, ensure_ascii=False)

In [22]:
# Tool 단독 실행 테스트
get_current_weather.invoke({
    "city" : "Seoul"
})

'{"found": true, "city": "서울특별시", "admin1": "서울특별시", "country": "대한민국", "latitude": 37.566, "longitude": 126.9784, "time": "2026-05-12T12:30", "weather": "부분적으로 흐림", "temperature_celsius": 18.9, "apparent_temperature_celsius": 20.5, "relative_humidity_percent": 83, "precipitation_mm": 0.0, "wind_speed_kmh": 3.1}'

## 날씨 Tool을 사용하는 Agent 만들기

In [23]:
weather_agent = create_agent(
    model=llm,
    tools=[get_current_weather],
    system_prompt="""
당신은 날씨 정보를 알려주는 도우미 AI입니다.

사용자가 현재 날씨를 물어보면 current_weather 도구를 사용하세요.
current_weather 도구를 호출할 때, city 값은 반드시 영어 도시명으로 변환해서 전달하세요.
예: 서울 -> Seoul, 부산 -> Busan, 도쿄 -> Tokyo, 뉴욕 -> Newyork

도구 결과에 있는 온도, 체감 온도, 습도, 강수량, 풍속 정보를 바탕으로 한국어로 답변하세요.
도구 결과에서 찾을 수 없는 정보는 지어내지 마세요.
"""
)

In [24]:
result = weather_agent.invoke({
    "messages":[
        {
            "role":"user",
            "content":"서울 현재 날씨 알려줘."
        }
    ]
})

result['messages'][-1].content

'서울의 현재 날씨는 부분적으로 흐림이며, 기온은 18.9도입니다. 체감 온도는 20.5도이고, 습도는 83%입니다. 강수량은 없으며, 바람은 시속 3.1km로 불고 있습니다.'

## Tavily 검색 Tool 사용하기

In [25]:
%pip install langchain-tavily

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
from langchain_tavily import TavilySearch

# Tavily 검색 Tool 준비
tavily_search = TavilySearch(
    max_results = 3,
    search_depth = "basic"
)

In [30]:
# Agent 연결 전 Tool 단독 호출 테스트
tavily_search.invoke({
    "query" : "LangChain create_agent 최신 활용법"
})

{'query': 'LangChain create_agent 최신 활용법',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/watch?v=Ipa6JNbFq4g',
   'title': 'LangChain/LangGraph V1 업데이트 이후 신기능 활용하여 에이전트 ...',
   'content': '하며, 에이전트 개발의 패러다임이 새롭게 정립되고 있습니다. 본 세션에서는 실무에서 바로 활용 가능한 최신 ... create_agent API를 통해',
   'score': 0.9999944,
   'raw_content': None},
  {'url': 'https://reference.langchain.com/python/langchain/agents/factory/create_agent',
   'title': 'create_agent - LangChain Reference Docs',
   'content': '##### LangChain. | `model`\\* | `str | BaseChatModel` | The language model for the agent. Can be a string identifier (e.g., `"openai:gpt-4"`) or a direct chat model instance (e.g., [`ChatOpenAI` or other another LangChain chat model). For a full list of supported model strings, see `init_chat_model`. | `tools` | `Sequence[BaseTool | Callable[..., Any] | dict[str, Any]] | None` | Default:`None` |. | `system_prompt` | `str | SystemMessage | None` | De

## LangChain Community Tool 사용하기
- LangChain 생태계에는 미리 만들어진 Tool이 많이 있다.
- Wikipidia Tool을 가져와서 사용해본다.

In [33]:
%pip install wikipedia

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11785 sha256=3f30e78980e73ea65a9f68f24e12ab4699fca09d5b123107bdfeabf5b4918d1d
  Stored in directory: c:\users\playdata\appdata\local\pip\cache\wheels\63\47\7c\a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

# Wikipedia 검색 Tool
wikipedia_tool = WikipediaQueryRun(
    api_wrapper = WikipediaAPIWrapper(
        top_k_results = 2,
        doc_content_chars_max = 800,
        lang = "ko"
    )
)

In [37]:
# Agent 연결 전 Tool 단독 실행
wikipedia_tool.invoke("LangChain")

'Page: 랭체인\nSummary: LangChain은 애플리케이션에 대형 언어 모델(LLM)의 통합을 돕는 소프트웨어 프레임워크이다. 언어 모델 통합 프레임워크로서 LangChain의 사용 사례는 문서 분석 및 자동 요약, 챗봇, 코드 분석 등 일반적인 언어 모델의 사용 사례와 크게 겹친다.\n\nPage: 프롬프트 엔지니어링\nSummary: 프롬프트 엔지니어링(Prompt Engineering)은 대규모 언어 모델(Large Language Models, LLMs)과 시각-언어 모델(Vision-Language Models, VLMs)의 출력을 최적화하기 위해 입력 프롬프트를 설계하고 구조화하는 기술이다. 모델 파라미터를 수정하지 않고도 특정 작업에 대한 모델의 성능을 향상시키는 것을 목표로 한다.'

## 여러 검색 Tool의 역할 구분

In [38]:
from datetime import datetime

today = datetime.now().strftime("%Y-%m-%d")

research_agent = create_agent(
    model=llm,
    tools=[tavily_search, wikipedia_tool],
    system_prompt=f"""
당신은 도구를 적절히 선택하는 AI 리서치 도우미입니다.
오늘 날짜는 {today}입니다.

도구 사용 기준:
1. 일반 개념이나 배경지식은 Wikipedia 도구를 사용하세요.
2. 최신 정보나 현재 상황 확인은 Tavily 검색 도구를 사용하세요.
3. 도구 결과에 없는 내용은 지어내지 마세요.
4. 최종 답변은 한국어로 간결하게 작성하세요.
"""
)

In [39]:
questions = [
    "RAG가 무엇인지 기본 개념을 설명해줘.",
    "RAG 관련 최근 기술 동향을 웹에서 찾아서 요약해줘."
]

for question in questions:
    print("질문 : ", question)
    result = research_agent.invoke({
        "messages" : [
            {
                "role" : "user",
                "content" : question
            }
        ]
    })
    print(result["messages"][-1].content)
    print("=" * 100)

질문 :  RAG가 무엇인지 기본 개념을 설명해줘.


c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\wikipedia\wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file c:\Users\Playdata\AppData\Local\miniforge3\envs\dl_nlp_env\Lib\site-packages\wikipedia\wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


RAG(검색 증강 생성, Retrieval-Augmented Generation)는 대형 언어 모델(LLM)이 새로운 정보를 검색하고 통합할 수 있도록 하는 기술입니다. 기존의 LLM은 훈련된 데이터에만 의존하지만, RAG는 데이터베이스나 문서, 웹 소스 등에서 관련 정보를 실시간으로 가져와 응답에 반영합니다. 이를 통해 최신 정보나 도메인 특화 정보를 활용할 수 있고, 잘못된 정보 생성(환각)을 줄이며, 응답에 출처를 포함해 신뢰성을 높일 수 있습니다. 또한, 새로운 데이터로 모델을 재훈련할 필요가 줄어들어 비용과 효율성 측면에서도 이점이 있습니다.
질문 :  RAG 관련 최근 기술 동향을 웹에서 찾아서 요약해줘.
2026년 RAG(Retrieval-Augmented Generation) 기술 동향 요약입니다.

- RAG는 단순히 벡터DB나 그래프DB 선택 문제가 아니라, 지식 접근을 '운영 가능한 시스템'으로 만드는 패턴 중심으로 진화 중입니다.
- 성능뿐 아니라 거버넌스, 보안, 최신성, 평가, 라우팅 등이 핵심 설계 요소가 되었습니다.
- 기업에서는 RAG가 내부 지식의 유통 경로 역할을 하며, permission-aware retrieval(권한 인지 검색)이 필수입니다.
- 실행형 RAG(툴/에이전트 결합)에서는 '틀린 행동' 문제에 대한 위험 관리가 중요해졌습니다.
- GraphRAG는 그래프DB 유행이 아니라 계층, 라우팅, 다중 홉 처리를 위한 조건부 고난도 옵션으로 자리잡고 있습니다.
- 최신 RAG는 텍스트를 넘어 이미지, 오디오, 비디오 등 멀티모달 데이터 처리로 확장되고 있습니다.

즉, 2026년 RAG는 단순 검색 보조를 넘어 운영 가능한 지식 인프라로 재편되고 있습니다.
